In [4]:
import sys

!{sys.executable} -m pip install \
langchain-community \
langchain-text-splitters \
langchain-groq \
langchain-huggingface \
sentence-transformers \
pypdf \
scikit-learn

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from sklearn.metrics.pairwise import cosine_similarity

from langchain_groq import ChatGroq

import numpy as np


class PDFChatbot:

    def __init__(self, pdf_path, groq_api_key):

        self.pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

        self.groq_api_key = "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"

        self.chat_history = []

        self.load_pdf()

        self.create_embeddings()

        self.load_llm()


    def load_pdf(self):

        loader = PyPDFLoader(self.pdf_path)

        documents = loader.load()

        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50
        )

        self.chunks = splitter.split_documents(documents)

        self.texts = [doc.page_content for doc in self.chunks]


    def create_embeddings(self):

        self.embedding_model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

        embeddings = self.embedding_model.embed_documents(
            self.texts
        )

        self.document_embeddings = np.array(embeddings)


    def load_llm(self):

        self.llm = ChatGroq(
            groq_api_key=self.groq_api_key,
            model_name="llama-3.3-70b-versatile"
        )


    def retrieve(self, query, k=3):

        query_embedding = self.embedding_model.embed_query(query)

        similarity_scores = cosine_similarity(
            [query_embedding],
            self.document_embeddings
        )[0]

        top_indices = similarity_scores.argsort()[::-1][:k]

        retrieved_chunks = [
            self.texts[i] for i in top_indices
        ]

        return retrieved_chunks


    def ask(self, question):

        retrieved_docs = self.retrieve(question)

        context = "\n".join(retrieved_docs)

        history_text = ""

        for message in self.chat_history:

            history_text += f"{message['role']}: {message['content']}\n"


        prompt = f"""
        You are a helpful PDF chatbot.

        Use the conversation history and document context
        to answer the user's question.

        Conversation History:
        {history_text}

        Context:
        {context}

        User Question:
        {question}
        """


        response = self.llm.invoke(prompt)

        answer = response.content


        self.chat_history.append({
            "role": "user",
            "content": question
        })

        self.chat_history.append({
            "role": "assistant",
            "content": answer
        })

        return answer


pdf_path = "/Users/Aryaanil/Documents/survey cdc.pdf"

groq_api_key = "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"


chatbot = PDFChatbot(
    pdf_path,
    groq_api_key
)


questions = [

    "What is the main topic of the PDF?",

    "Can you explain that topic in simpler words?",

    "Which tools or libraries are related to it?",

    "How are those libraries used in the document?",

    "Summarize our conversation in a few lines."
]


for i, question in enumerate(questions, start=1):

    answer = chatbot.ask(question)

    print(f"\nTURN {i}")

    print("\nUSER:")
    print(question)

    print("\nBOT:")
    print(answer)

    print("\n" + "=" * 70)


print("\nFULL MESSAGE HISTORY")


for message in chatbot.chat_history:

    print(f"\n{message['role'].upper()}:")
    print(message['content'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


TURN 1

USER:
What is the main topic of the PDF?

BOT:
The main topic of the PDF appears to be related to academic and student-related matters, specifically focusing on the student community, with an emphasis on stress and how students handle difficult situations.


TURN 2

USER:
Can you explain that topic in simpler words?

BOT:
The PDF is talking about how students feel and deal with stress. It says that most students experience stress sometimes or often, but they also feel happy in their daily lives. Only a small number of students said they rarely or never experience stress. So, stress is a common problem for many students, but they are generally happy overall.


TURN 3

USER:
Which tools or libraries are related to it?

BOT:
The PDF doesn't explicitly mention specific tools or libraries related to the topic of student stress and happiness. However, based on the context of the survey and the department names mentioned (e.g., Department of Computer Applications), it's possible that